In [ ]:
# Install dependencies
!pip install emoji nltk keras_tuner wordsegment magnitude datasets

In [ ]:
# Set mixed precision policy
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')


In [ ]:
# Imports
import pandas as pd
import numpy as np
import re
import os
import pickle
import html
import nltk
import emoji
import keras_tuner as kt
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils import class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GRU, Layer, MaxPooling1D, GlobalMaxPooling1D, Conv1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Callback, TensorBoard, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras import backend as K
from nltk.tokenize import TweetTokenizer
from tqdm import tqdm
from wordsegment import load, segment
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split


load()
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

In [ ]:
# Download Glove Embeddings and Unzip them
if not os.path.exists('glove.twitter.27B.100d.txt'):
    !wget https://nlp.stanford.edu/data/glove.twitter.27B.zip
    !unzip glove.twitter.27B.zip
    print("Embeddings Downloaded and Unzipped!")
else:
    print("Embeddings already exist!")

In [ ]:
# --- CONFIGURATION ---
GLOVE_FILE = 'glove.twitter.27B.100d.txt'
EMBEDDING_DIM = 100
SAVE_DIR = 'Models'
SNAPSHOT_DIR = 'snapshot_data'
N_REPEATS = 2
EPOCHS = 100
BATCH_SIZE = 32
CONTRACTIONS = {
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "won't": "will not",
    "wouldn't": "would not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "mightn't": "might not",
    "mustn't": "must not"
}
CALLBACKS = [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1), TensorBoard(log_dir='logs', histogram_freq=1), ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)]

In [ ]:
# Create directory for models if it doesn't exist
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)



In [ ]:
# get_berkeley_class: Bins Continuous Values into Discrete Classes
def get_berkeley_class(raw_score):
    if raw_score > 0.50:
        return 2  # Hate Speech
    elif raw_score < -1.00:
        return 0  # Supportive / Safe
    else:
        return 1  # The "Gray Area" (Neutral / Offensive)

In [ ]:
# load_data: Loads data from dataset
def load_data():
    # --- STEP 1: LOAD DATA ---
    ds = load_dataset("ucberkeley-dlab/measuring-hate-speech")
    df = ds['train'].to_pandas()
    trimmed_df = df.groupby('text').agg({'hate_speech_score':'mean'}).reset_index()
    trimmed_df['label'] = trimmed_df['hate_speech_score'].apply(get_berkeley_class)
    return trimmed_df

In [ ]:
# Shared historical text-cleaning contract (used by notebook and inference).


In [ ]:
# Shared historical text-cleaning contract (used by notebook and inference).


In [ ]:
# Shared historical text-cleaning contract (used by notebook and inference).


In [ ]:
from preprocessing import clean_text as clean_text_for_twitter, encode_tokens


In [ ]:
# PreprocessData:  Loads data from file, and performs preprocessing steps to prepare for embeddings mapping.
def PreprocessData():
    tqdm.pandas()
    # --- STEP 1: DATA LOADING ---
    final_df = load_data()
    # --- Export file for transparency
    final_df.to_csv('final_df.csv', index=False)
    # --- STEP 2: PREPROCESSING ---
    

    final_df['clean_text'] = final_df['text'].progress_apply(clean_text_for_twitter)



    final_df.head()
    # Check the distribution of lengths in your filtered 500 tweets
    tweet_lengths = final_df['clean_text'].apply(lambda x: len(str(x).split()))

    print(f"Average Length: {tweet_lengths.mean():.1f} words")
    print(f"Maximum Length: {tweet_lengths.max()} words")
    print(f"95th Percentile: {tweet_lengths.quantile(0.95):.1f} words")
    print(f"99th Percentile: {tweet_lengths.quantile(0.99):.1f} words")
    MAX_LEN = int(tweet_lengths.quantile(0.99))

    # --- STEP 3: TOKENIZATION ---
    vocab = set()
    for text in tqdm(final_df['clean_text'], desc="Building Vocabulary"):
        vocab.update(text)

    word_index = {word: idx for idx, word in enumerate(vocab, start=1)}
    print(f"Unique Tokens: {len(word_index)}")


    sequences = []
    for text in tqdm(final_df['clean_text'], desc="Tokenizing"):
        sequences.append([word_index[word] for word in text])
        


    X = np.asarray([encode_tokens(tokens, word_index, MAX_LEN) for tokens in final_df["clean_text"]], dtype=np.int32)
    y = final_df['label'].values
    # --- STEP 4: LOAD GLOVE ---
    print("Loading GloVe Embeddings...")
    embedding_index = {}
    with open(GLOVE_FILE, encoding='utf-8') as f:
        for line in tqdm(f, desc="Loading GloVe Embeddings"):
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embedding_index[word] = coefs

    num_words = len(word_index) +1
    embedding_matrix = np.zeros((num_words, EMBEDDING_DIM))
    hits = 0
    misses = 0
    rescued_strip = 0
    rescued_seg = 0
    missing_words = []
    for word, i in tqdm(word_index.items(), desc="Building Embedding Matrix"):
        flag = False
        embedding_vector = embedding_index.get(word)
        if embedding_vector is None and word.startswith("#") and len(word) > 1:
            flag=True
            clean_word = word[1:]  # Remove the leading '#'
            embedding_vector = embedding_index.get(clean_word)
            if embedding_vector is not None:
                rescued_strip += 1
        if embedding_vector is None and len(word) > 1:
            if flag:
                word = clean_word
            subwords = segment(word)
            if len(subwords) > 0:
                sub_vectors = []
                for sub in subwords:
                    v = embedding_index.get(sub)
                    if v is not None:
                        sub_vectors.append(v)
                if len(sub_vectors) > 0:
                    embedding_vector = np.mean(sub_vectors, axis=0)
                    rescued_seg += 1
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1
        else:
            misses += 1
            missing_words.append(word)

    saturation = (hits / (hits + misses)) * 100
    print(f"\nSaturation: {saturation:.2f}%")
    print(f"Successful Matches: {hits}")
    print(f"Missed Tokens: {misses}")
    print(f"Rescued Tokens (Strip): {rescued_strip}")
    print(f"Rescued Tokens (Segment): {rescued_seg}")
    print("\nTop 10 Missed Words (Why are we failing?):")
    print(missing_words[:10])


    if not os.path.exists(SNAPSHOT_DIR):
        os.makedirs(SNAPSHOT_DIR)

    print("Saving snapshot to disk...")

    # 1. Save the Processed Tensors (X and y)
    np.save(f'{SNAPSHOT_DIR}/X.npy', X)
    np.save(f'{SNAPSHOT_DIR}/y.npy', y)

    # 2. Save the Expensive Embedding Matrix
    np.save(f'{SNAPSHOT_DIR}/embedding_matrix.npy', embedding_matrix)

    # 3. Save the Dictionary (Crucial for inference later)
    with open(f'{SNAPSHOT_DIR}/word_index.pickle', 'wb') as handle:
        pickle.dump(word_index, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
    print("Snapshot saved! You can skip processing next time.")
    return X, y, embedding_matrix, word_index, MAX_LEN, EMBEDDING_DIM, missing_words

In [ ]:
# Check if we have a save file already on file. If not, then create embeddings map and data numpy saves for faster training in future
if os.path.exists(f'{SNAPSHOT_DIR}/X.npy'):
    print("Loading data from Snapshot (Fast Mode)...")
    
    # Load Numpy Arrays
    X = np.load(f'{SNAPSHOT_DIR}/X.npy')
    y = np.load(f'{SNAPSHOT_DIR}/y.npy')
    embedding_matrix = np.load(f'{SNAPSHOT_DIR}/embedding_matrix.npy')
    
    # Load Dictionary
    with open(f'{SNAPSHOT_DIR}/word_index.pickle', 'rb') as handle:
        word_index = pickle.load(handle)
        
    # Re-calculate MAX_LEN and EMBEDDING_DIM from the loaded data
    MAX_LEN = X.shape[1]
    EMBEDDING_DIM = embedding_matrix.shape[1]
    
    print(f"Loaded {len(X)} rows. Max Len: {MAX_LEN}. Vocab: {len(word_index)}")
    print("Ready for Training.")

else:
    print("Snapshot not found. Running Full Processing...")
    X, y, embedding_matrix, word_index, MAX_LEN, EMBEDDING_DIM, missing_words = PreprocessData()

In [ ]:
# Export Prepared Data to CSV
export_df = pd.DataFrame(X)
export_df['Label'] = y
export_df.to_csv('processed_data.csv', index=False)
print("Exported X and y to processed_data.csv")


In [ ]:
from model_layers import MaskedGlobalAverageTopKPooling1D


In [ ]:
# Hypermodel Function to define structure of the model
def hypermodel(hp):
    model = Sequential()
    model.add(Embedding(len(word_index) + 1, EMBEDDING_DIM, 
                            weights=[embedding_matrix], 
                            trainable=False, mask_zero=True))
    model.add(Bidirectional(GRU(units=100, 
                                dropout=hp.Float('gru_drop',min_value=0.0 ,max_value=0.5,step=0.1), 
                                return_sequences=True, 
                                recurrent_dropout=0.0))) 
    model.add(MaskedGlobalAverageTopKPooling1D(k=hp.Int('pool_k', min_value=2, max_value=5, step=1)))
    model.add(Dense(hp.Int('dense_units', min_value=32, max_value=128, step=32), activation='relu'))
    model.add(Dropout(hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)))
    model.add(Dense(3, activation='softmax', dtype='float32'))
    lr = hp.Float('learning_rate', min_value=1e-5, max_value=5e-3, sampling='log')
    model.compile(optimizer=Adam(learning_rate=lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Train-Val-Test Split
x_train_val, x_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)

In [ ]:
# Determine Class Weights
# Calculate weights automatically based on the counts
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary for Keras
class_weight_dict = dict(enumerate(weights))

print("Class Weights:", class_weight_dict)
# Expected: {0: ~2.0, 1: ~0.5, 2: ~2.0}
# Meaning: Pay 4x more attention to Safe/Hate than the middle class.

In [ ]:
# Create datasets
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))


In [ ]:
# Buffer for training and validation datasets
train_dataset = (train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE))

In [ ]:
# Perform Keras Tuner Search for best hyperparameters for final training.

tuner = kt.Hyperband(hypermodel=hypermodel, objective='val_loss', max_epochs=10, factor=3, directory='KerasTuner', project_name='best_model')
tuner.search(train_dataset, validation_data=val_dataset, epochs=10, class_weight=class_weight_dict)
best_hps = tuner.get_best_hyperparameters()[0]
print(best_hps.values)

In [ ]:
# Instantiate the model
model = hypermodel(best_hps)

In [ ]:
# Perform actual training and save completed model
model.fit(train_dataset, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=val_dataset, callbacks=CALLBACKS, class_weight=class_weight_dict)


In [ ]:
# Unfreeze and Fine-Tune

model.layers[0].trainable = True # Unfreeze the Embedding Layer

# Recompile with a tiny learning rate (10x smaller)
model.compile(
    optimizer=Adam(learning_rate=best_hps.get('learning_rate')/10), 
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

# Train for a few more epochs
history_fine = model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, class_weight=class_weight_dict, callbacks=CALLBACKS, batch_size=BATCH_SIZE)
model.save(f'{SAVE_DIR}/model.keras')

In [ ]:
# Generate Completed Model Summary
model.summary()

In [ ]:
# Get the detailed breakdown
print("Generating test predictions...")
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test

print(f"Accuracy: {accuracy_score(y_true, y_pred_classes):.2%}")
print(f"Precision: {precision_score(y_true, y_pred_classes, average='macro'):.2%}")
print(f"Recall: {recall_score(y_true, y_pred_classes, average='macro'):.2%}")

print(classification_report(y_true, y_pred_classes, target_names=['Safe', 'Offensive', 'Hate']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)

# Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Safe (0)', 'Offensive (1)', 'Hate (2)'],
            yticklabels=['Safe (0)', 'Offensive (1)', 'Hate (2)'])

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Final Test Set Confusion Matrix', fontsize=14)
plt.show()